In [1]:
import pandas as pd
import cupy as cp
import cudf
import cuml
import torch
import gc
import time
from cuml.metrics import mean_squared_error, mean_squared_log_error, median_absolute_error, r2_score, accuracy_score, confusion_matrix, kl_divergence
from cuml.metrics import log_loss, roc_auc_score, nan_euclidean_distances, pairwise_distances, sparse_pairwise_distances
from cuml.model_selection import train_test_split, KFold
from cuml.linear_model import LogisticRegression, MBSGDClassifier
from cuml.multiclass import OneVsRestClassifier, OneVsOneClassifier 
from cuml.neighbors import KNeighborsClassifier, KNeighborsRegressor
from cuml.naive_bayes import BernoulliNB, CategoricalNB, ComplementNB, GaussianNB, MultinomialNB
from cuml import LinearRegression, Ridge
from cuml.linear_model import Lasso, ElasticNet

In [2]:
df = cudf.read_csv('heart_disease_health_indicators_BRFSS2015.csv')
df

,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253675,0.0,1.0,1.0,1.0,45.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,5.0,0.0,1.0,5.0,6.0,7.0
253676,0.0,1.0,1.0,1.0,18.0,0.0,0.0,2.0,0.0,0.0,...,1.0,0.0,4.0,0.0,0.0,1.0,0.0,11.0,2.0,4.0
253677,0.0,0.0,0.0,1.0,28.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,5.0,2.0
253678,0.0,1.0,0.0,1.0,23.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,7.0,5.0,1.0


In [ ]:
from ipynb.fs.full.Normalization_MultiClass_to_import import *

In [ ]:
from ipynb.fs.full.GeneratedDataMultiClass_to_import import *

In [ ]:
torch.cuda.empty_cache()
gc.collect()

In [ ]:
class MultiClass(object):
    def __init__(self, dataset, generated_data_multiclass_global_1, estimator : str):
        self.dataset = dataset.copy().reset_index(drop = True)
        self.X = cudf.DataFrame(self.dataset.copy().drop(['GenHlth'], axis = 1))
        self.y = cudf.DataFrame(self.dataset['GenHlth'].copy())
        self.generated_data_multiclass_global_1 = generated_data_multiclass_global_1.copy().reset_index(drop = True)
        self.dataset_numpy_array = self.dataset.copy().to_numpy()
        self.X_cupy_array = self.X.to_cupy()
        self.y_cupy_array = self.y.to_cupy()
        self.gen_data_cupy_array = self.generated_data_multiclass_global_1.to_cupy()
        self.estimator = estimator


    def train_test_split(self):
        global X_train_global
        global X_test_global
        global y_train_global
        global y_test_global

        X_train, X_test, y_train, y_test = train_test_split(self.X_cupy_array, self.y_cupy_array, test_size=0.2, random_state=42)
        X_train_global = X_train
        X_test_global = X_test
        y_train_global = y_train
        y_test_global = y_test

    def OneVsOneClassifier(self):
        global model_ovoc_global 
        global ovoc_global
        global ovoc_predict_test_global
        global ovoc_predict_global_1
        global mean_squared_error_global_ovoc
        global mean_squared_log_error_global_ovoc
        global median_absolute_error_global_ovoc
        global r2_score_global_ovoc
        global accuracy_score_global_ovoc
        global kl_divergence_global_ovoc
        global log_loss_global_ovoc
        global roc_auc_score_global_ovoc
        global nan_euclidean_distances_global_ovoc
        global pairwise_distances_global_ovoc
        global sparse_pairwise_distances_global_ovoc

        if (self.estimator == 'LogisticRegression'):
            model = OneVsOneClassifier(LogisticRegression(penalty='l2', tol=0.0001, C=1.0, fit_intercept=True, class_weight=None, 
                                                          max_iter=1000, linesearch_max_iter=50, l1_ratio=None, solver='qn', 
                                                          lbfgs_memory=5, penalty_normalized=True, verbose=False, 
                                                          output_type=None))
            model_ovoc_global = model
        elif (self.estimator == 'LinearRegression'):
            model = OneVsOneClassifier(LinearRegression(algorithm='auto', fit_intercept=True, copy_X=True, verbose=False, 
                                                          output_type=None))
            model_ovoc_global = model
        elif (self.estimator == 'Ridge'):
            model = OneVsOneClassifier(Ridge(alpha=1e-5))
            model_ovoc_global = model
        elif (self.estimator == 'Lasso'):
            model = OneVsOneClassifier(Lasso(alpha = 0.1, fit_intercept=True, max_iter=1000, tol=0.001, solver='cd', 
                                             selection='cyclic', handle=None, output_type=None, verbose=False))
            model_ovoc_global = model
        elif (self.estimator == 'ElasticNet'):
            model = OneVsOneClassifier(ElasticNet(alpha = 0.1, l1_ratio=0.5, solver='qn'))
            model_ovoc_global = model
            
        ovoc = model_ovoc_global.fit(X_train_global.copy(), y_train_global.copy().ravel())
        ovoc_global = ovoc
        preds_test = model_ovoc_global.predict(X_test_global.copy())
        ovoc_predict_test_global = cudf.DataFrame(preds_test.copy(), columns = ['GenHlth']).reset_index(drop = True)
        
        mean_squared_error_global_ovoc = mean_squared_error(y_test_global.copy(), preds_test)
        #mean_squared_log_error_global_ovoc = mean_squared_log_error(y_test_global.copy(), preds_test)
        median_absolute_error_global_ovoc = median_absolute_error(y_test_global.copy(), preds_test)
        r2_score_global_ovoc = r2_score(y_test_global.copy(), preds_test)
        accuracy_score_global_ovoc = accuracy_score(y_test_global.copy(), preds_test)
        kl_divergence_global_ovoc = kl_divergence(y_test_global.copy(), preds_test)
        #log_loss_global_ovoc = log_loss(y_test_global.copy(), preds_test)
        roc_auc_score_global_ovoc = roc_auc_score(y_test_global.copy(), preds_test)
        #nan_euclidean_distances_global_ovoc = nan_euclidean_distances(y_test_global.copy(), preds_test)
        #pairwise_distances_global_ovoc = pairwise_distances(y_test_global.copy(), preds_test)
        #sparse_pairwise_distances_global_ovoc = sparse_pairwise_distances(y_test_global.copy(), preds_test)
     
        preds_1 = model_ovoc_global.predict(self.gen_data_cupy_array)
        ovoc_predict_global_1 = cudf.DataFrame(preds_1.copy(), columns = ['GenHlth']).reset_index(drop = True) 

    def OneVsRestClassifier(self):
        global model_ovrc_global
        global ovrc_global
        global ovrc_predict_test_global
        global ovrc_predict_global_1
        global mean_squared_error_global_ovrc
        global mean_squared_log_error_global_ovrc
        global median_absolute_error_global_ovrc
        global r2_score_global_ovrc
        global accuracy_score_global_ovrc
        global kl_divergence_global_ovrc
        global log_loss_global_ovrc
        global roc_auc_score_global_ovrc
        global nan_euclidean_distances_global_ovrc
        global pairwise_distances_global_ovrc
        global sparse_pairwise_distances_global_ovrc

        if (self.estimator == 'LogisticRegression'):
            model = OneVsRestClassifier(LogisticRegression(penalty='l2', tol=0.0001, C=1.0, fit_intercept=True, class_weight=None, 
                                                          max_iter=1000, linesearch_max_iter=50, l1_ratio=None, solver='qn', 
                                                          lbfgs_memory=5, penalty_normalized=True, verbose=False, 
                                                          output_type=None))
            model_ovrc_global = model
        elif (self.estimator == 'LinearRegression'):
            model = OneVsRestClassifier(LinearRegression(algorithm='auto', fit_intercept=True, copy_X=True, verbose=False, 
                                                          output_type=None))
            model_global = model
        elif (self.estimator == 'Ridge'):
            model = OneVsRestClassifier(Ridge(alpha=1e-5))
            model_ovrc_global = model
        elif (self.estimator == 'Lasso'):
            model = OneVsRestClassifier(Lasso(alpha = 0.1, fit_intercept=True, max_iter=1000, tol=0.001, solver='cd', 
                                             selection='cyclic', handle=None, output_type=None, verbose=False))
            model_ovrc_global = model
        elif (self.estimator == 'ElasticNet'):
            model = OneVsRestClassifier(ElasticNet(alpha = 0.1, l1_ratio=0.5, solver='qn'))
            model_ovrc_global = model
            
        ovrc = model_ovrc_global.fit(X_train_global.copy(), y_train_global.copy().ravel())
        ovrc_global = ovrc
        preds_test = model_ovrc_global.predict(X_test_global.copy())
        ovrc_predict_test_global = cudf.DataFrame(preds_test.copy(), columns = ['GenHlth']).reset_index(drop = True)
        
        mean_squared_error_global_ovrc = mean_squared_error(y_test_global.copy(), preds_test)
        #mean_squared_log_error_global_ovrc = mean_squared_log_error(y_test_global.copy(), preds_test)
        median_absolute_error_global_ovrc = median_absolute_error(y_test_global.copy(), preds_test)
        r2_score_global_ovrc = r2_score(y_test_global.copy(), preds_test)
        accuracy_score_global_ovrc = accuracy_score(y_test_global.copy(), preds_test)
        kl_divergence_global_ovrc = kl_divergence(y_test_global.copy(), preds_test)
        #log_loss_global_ovrc = log_loss(y_test_global.copy(), preds_test)
        roc_auc_score_global_ovrc = roc_auc_score(y_test_global.copy(), preds_test)
        #nan_euclidean_distances_global_ovrc = nan_euclidean_distances(y_test_global.copy(), preds_test)
        #pairwise_distances_global_ovrc = pairwise_distances(y_test_global.copy(), preds_test)
        #sparse_pairwise_distances_global_ovrc = sparse_pairwise_distances(y_test_global.copy(), preds_test)
     
        preds_1 = model_ovrc_global.predict(self.gen_data_cupy_array)
        ovrc_predict_global_1 = cudf.DataFrame(preds_1.copy(), columns = ['GenHlth']).reset_index(drop = True)

    def main(self):
        st = time.time()
        self.train_test_split()
        self.OneVsOneClassifier()
        self.OneVsRestClassifier()
        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')

In [ ]:
class Metrics(object):
    def OneVsOneClassifier(self):
        print('OneVsOneClassifier: ')
        print('Mean squared error: ')
        print(mean_squared_error_global_ovoc)
        print('\n')
        print('Median Absolute Error: ')
        print(median_absolute_error_global_ovoc)
        print('\n')
        print('R2 Score: ')
        print(r2_score_global_ovoc)
        print('\n')
        print('Accuracy Score: ')
        print(accuracy_score_global_ovoc)
        print('\n')
        print('Kl Divergence: ')
        print(kl_divergence_global_ovoc)
        print('\n')
        print('ROC AUC Score: ')
        print(roc_auc_score_global_ovoc)
        print('\n')

    def OneVsRestClassifier(self):
        print('OneVsRestClassifier: ')
        print('Mean squared error: ')
        print(mean_squared_error_global_ovrc)
        print('\n')
        print('Median Absolute Error: ')
        print(median_absolute_error_global_ovrc)
        print('\n')
        print('R2 Score: ')
        print(r2_score_global_ovrc)
        print('\n')
        print('Accuracy Score: ')
        print(accuracy_score_global_ovrc)
        print('\n')
        print('Kl Divergence: ')
        print(kl_divergence_global_ovrc)
        print('\n')
        print('ROC AUC Score: ')
        print(roc_auc_score_global_ovrc)
        print('\n')

    def main(self):
        self.OneVsOneClassifier()
        self.OneVsRestClassifier()

In [ ]:
torch.cuda.empty_cache()
gc.collect()

In [44]:
multiclass_binarizer_LogisticRegression = MultiClass(binarizer_merged_multiclass_global, generated_data_multiclass_global,
                                 'LogisticRegression')
multiclass_binarizer_LogisticRegression.main()
multiclass_binarizer_LogisticRegression_metrics = Metrics()
print("Metrics for binarizer normalized data is ready. Run 'multiclass_binarizer_LogisticRegression_metrics.main()'!")
#multiclass_binarizer_LogisticRegression_metrics.main()

Execution time: 1.2585530281066895 seconds
Metrics for binarizer normalized data is ready. Run 'multiclass_binarizer_LogisticRegression_metrics.main()'!


In [45]:
multiclass_binarizer_LinearRegression = MultiClass(binarizer_merged_multiclass_global, generated_data_multiclass_global,
                                 'LinearRegression')
multiclass_binarizer_LinearRegression.main()
multiclass_binarizer_LinearRegression_metrics = Metrics()
print("Metrics for binarizer normalized data is ready. Run 'multiclass_binarizer_LinearRegression_metrics.main()'!")
#multiclass_binarizer_LinearRegression_metrics.main()

Execution time: 1.7533836364746094 seconds
Metrics for binarizer normalized data is ready. Run 'multiclass_binarizer_LinearRegression_metrics.main()'!


In [46]:
multiclass_binarizer_Ridge = MultiClass(binarizer_merged_multiclass_global, generated_data_multiclass_global,
                                 'Ridge')
multiclass_binarizer_Ridge.main()
multiclass_binarizer_Ridge_metrics = Metrics()
print("Metrics for binarizer normalized data is ready. Run 'multiclass_binarizer_Ridge_metrics.main()'!")
#multiclass_binarizer_Ridge_metrics.main()

Execution time: 2.743795156478882 seconds
Metrics for binarizer normalized data is ready. Run 'multiclass_binarizer_Ridge_metrics.main()'!


In [47]:
multiclass_binarizer_Lasso = MultiClass(binarizer_merged_multiclass_global, generated_data_multiclass_global,
                                 'Lasso')
multiclass_binarizer_Lasso.main()
multiclass_binarizer_Lasso_metrics = Metrics()
print("Metrics for binarizer normalized data is ready. Run 'multiclass_binarizer_Lasso_metrics.main()'!")
#multiclass_binarizer_Lasso_metrics.main()

MemoryError: std::bad_alloc: out_of_memory: CUDA error (failed to allocate 3200000 bytes) at: /tmp/pip-build-env-n3_q0rui/normal/lib/python3.10/site-packages/librmm/include/rmm/mr/cuda_memory_resource.hpp:51: cudaErrorMemoryAllocation out of memory

In [ ]:
multiclass_binarizer_ElasticNet = MultiClass(binarizer_merged_multiclass_global, generated_data_multiclass_global,
                                 'ElasticNet')
multiclass_binarizer_ElasticNet.main()
multiclass_binarizer_ElasticNet_metrics = Metrics()
print("Metrics for binarizer normalized data is ready. Run 'multiclass_binarizer_ElasticNet_metrics.main()'!")
#multiclass_binarizer_ElasticNet_metrics.main()

In [ ]:
torch.cuda.empty_cache()
gc.collect()